# 11 · Modelo por municipio — GradientBoosting

Un GradientBoosting **por municipio** (67 modelos) con features externas + lag.

Features: `anio` (tendencia), `iph_media`, `iph_max` (isla), `ocupacion_media` (municipio), `lluvia_anual_mm` (municipio), `lag1` (consumo del anio anterior).

- Ventana: 2015+ (limitada por la lluvia); train 2016-2021, test 2022-2024 (prediccion recursiva: el lag se actualiza con la prediccion del anio anterior)
- Resultados: `results/11_modelo_municipio_metrics.csv`


In [1]:
import polars as pl
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted")
DATA = Path("data")
RESULTS = Path("results")
RESULTS.mkdir(exist_ok=True)

# --- Datos
abast = pl.read_csv(DATA / "abastecimiento_urbano_baleares.csv", infer_schema_length=None).select(["cod_municipio", "anio", "consumo_hm3"])
presion = pl.read_csv(DATA / "presion_humana.csv", infer_schema_length=None)
ocup = pl.read_csv(DATA / "ocupacion_turistica.csv", infer_schema_length=None)
lluvia = pl.read_csv(DATA / "lluvia_masa_subterranea.csv", infer_schema_length=None)
mma = pl.read_csv(DATA / "municipio_masa_subterranea.csv", infer_schema_length=None)
mun = pl.read_csv(DATA / "municipio.csv", infer_schema_length=None).select(["cod_municipio", "cod_provincia", "nombre_municipio"])

# Isla IPH: 071 (Formentera) y 072 (Eivissa) comparten serie NUTS
isla_map = pl.DataFrame({
    "cod_provincia": [71, 72, 73, 74],
    "isla": ["Eivissa i Formentera", "Eivissa i Formentera", "Mallorca", "Menorca"],
})

# --- Features
iph = (presion.group_by(["nombre_isla", "anio"])
       .agg(iph_media=pl.col("iph").mean(), iph_max=pl.col("iph").max()))
ocup_m = (ocup.group_by(["cod_municipio_ine", "anio"])
          .agg(ocupacion_media=pl.col("ocupacion_plazas_pct").mean()))
ll_m = (lluvia.group_by(["cod_masa", "anio"])
        .agg(lluvia_anual_mm=pl.col("precipitacion_mm").sum())
        .join(mma, on="cod_masa")
        .group_by(["cod_municipio", "anio"])
        .agg(lluvia_anual_mm=pl.col("lluvia_anual_mm").mean()))

panel = (
    abast
    .join(mun, on="cod_municipio", how="left")
    .join(isla_map, on="cod_provincia", how="left")
    .join(iph, left_on=["isla", "anio"], right_on=["nombre_isla", "anio"], how="left")
    .join(ocup_m, left_on=["cod_municipio", "anio"], right_on=["cod_municipio_ine", "anio"], how="left")
    .join(ll_m, on=["cod_municipio", "anio"], how="left")
    .select([
        "cod_municipio", "nombre_municipio", "isla", "anio", "consumo_hm3",
        "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm",
    ])
    .with_columns(
        pl.col("ocupacion_media").fill_null(0.0),
        pl.col("lluvia_anual_mm").fill_null(pl.col("lluvia_anual_mm").mean()),
    )
)

TEST_START = 2022
FEATURES = ["anio", "iph_media", "iph_max", "ocupacion_media", "lluvia_anual_mm", "lag1"]

panel = (
    panel
    .filter(pl.col("anio") >= 2015)
    .sort(["cod_municipio", "anio"])
    .with_columns(lag1=pl.col("consumo_hm3").shift(1).over("cod_municipio"))
)
print("panel:", panel.shape)
panel.head(8)


panel: (670, 10)


cod_municipio,nombre_municipio,isla,anio,consumo_hm3,iph_media,iph_max,ocupacion_media,lluvia_anual_mm,lag1
i64,str,str,i64,f64,f64,i64,f64,f64,f64
7001,"""Alaró""","""Mallorca""",2015,0.255,1.0826e6,1388045,0.0,408.1,null
7001,"""Alaró""","""Mallorca""",2016,0.278,1.1121e6,1415443,0.0,384.1,0.255
7001,"""Alaró""","""Mallorca""",2017,0.277,1.13149e6,1434094,0.0,496.2,0.278
7001,"""Alaró""","""Mallorca""",2018,0.263,1.1359e6,1414410,0.0,762.0,0.277
7001,"""Alaró""","""Mallorca""",2019,0.279,1.1458e6,1424487,0.0,457.0,0.263
7001,"""Alaró""","""Mallorca""",2020,0.277,981758.416667,1097115,0.0,584.7,0.279
7001,"""Alaró""","""Mallorca""",2021,0.289,1.0554e6,1261233,0.0,620.6,0.277
7001,"""Alaró""","""Mallorca""",2022,0.294,1149634.5,1422758,0.0,455.8,0.289


In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def metricas(test, pred):
    test, pred = np.asarray(test, float), np.asarray(pred, float)
    return dict(
        mae=float(mean_absolute_error(test, pred)),
        mape=float(np.mean(np.abs((test - pred) / test)) * 100),
        rmse=float(mean_squared_error(test, pred) ** 0.5),
        r2=float(r2_score(test, pred)),
    )

def particiones(g):
    g = g.sort("anio")
    train = g.filter(pl.col("anio") < TEST_START)
    test = g.filter(pl.col("anio") >= TEST_START)
    return train, test


In [3]:
from sklearn.ensemble import GradientBoostingRegressor

def predict_recursivo(model, train_df, test_df):
    """Predice test anio a anio actualizando lag1 con la prediccion (sin fuga de datos)."""
    hist = train_df.sort("anio")
    if hist.height < 2:
        return [float(hist["consumo_hm3"].mean())] * test_df.height
    last = float(hist["consumo_hm3"].to_list()[-1])
    preds = []
    for row in test_df.sort("anio").iter_rows(named=True):
        x = [float(row[f]) for f in FEATURES]
        x[FEATURES.index("lag1")] = last
        p = max(float(model.predict([x])[0]), 0.0)
        preds.append(p)
        last = p
    return preds


In [4]:
# ── Entrenar 67 modelos (uno por municipio) y evaluar
filas = []
skipped = []
for cod_t, g in panel.partition_by("cod_municipio", as_dict=True).items():
    cod = int(cod_t[0])
    train, test = particiones(g)
    train = train.drop_nulls(subset=["lag1"])  # 2015 no tiene lag1
    if train.height < 4:
        skipped.append(cod)
        pred = [float(train["consumo_hm3"].mean())] * test.height
    else:
        model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=2, random_state=42)
        model.fit(train.select(FEATURES).to_numpy(), train["consumo_hm3"].to_numpy())
        pred = predict_recursivo(model, train, test)
    filas.append({"modelo": "gb_municipio", "cod_municipio": cod,
                  **metricas(test["consumo_hm3"].to_list(), pred)})

print("municipios con train < 4 filas (media historica):", skipped)
res = pd.DataFrame(filas)
res.to_csv(RESULTS / "11_modelo_municipio_metrics.csv", index=False)
print(res[["mae", "mape", "rmse", "r2"]].mean().round(3))
print(res[["mae", "mape", "rmse", "r2"]].median().round(3))


municipios con train < 4 filas (media historica): []
mae      0.121
mape     8.685
rmse     0.131
r2     -21.488
dtype: float64
mae     0.039
mape    7.312
rmse    0.049
r2     -5.594
dtype: float64


In [5]:
# ── Comparacion con el mejor baseline (naive)
base = pd.read_csv(RESULTS / "10_baseline_metrics.csv")
naive = base[base["modelo"] == "naive"][["cod_municipio", "mape"]].rename(columns={"mape": "mape_naive"})
cmp = res.merge(naive, on="cod_municipio")
print("municipios donde gb_municipio mejora al naive (MAPE):", (cmp["mape"] < cmp["mape_naive"]).sum(), "de", cmp.shape[0])
print("MAPE medio gb_municipio:", round(cmp["mape"].mean(), 2), "| naive:", round(cmp["mape_naive"].mean(), 2))


municipios donde gb_municipio mejora al naive (MAPE): 44 de 67
MAPE medio gb_municipio: 8.68 | naive: 11.56


**Conclusiones**

- 67 modelos independientes: capturan la dinamica propia de cada municipio, pero con solo 6-7 anios de entrenamiento (2016-2021) el riesgo de overfitting es alto -> arboles poco profundos (depth=2).
- La comparacion con `naive` dice si las features externas (IPH/ocupacion/lluvia) anaden senal o solo ruido.
